# NB13 The Ethereum Virtual Machine and gas — Chapter 13

A chapter notebook for *Blockchain for Lawyers: A Reading Through Technology and Economics*. The code is the same as in the text.

- All code here is a **toy implementation** meant to make a principle visible. Do not use it with real assets, real private keys or real transactions.
- No external API is called, and everything runs without a network. Cells that need the bundled snapshot data carry that data inside the cell.
- Standard library only (run and checked on Python 3.13). There is no install cell.
- The outputs are the editor's own run, stored as produced. Running the cells again should give the same values (random seeds are fixed).

In [ ]:
# NB13-1: toy EVM — runs bytecode one instruction at a time, deducting gas (for study; not for real use)
# The fee schedule values are copied unchanged from Appendix G of the Yellow Paper (Shanghai version). The instruction set and the storage cost rules are greatly simplified.
G_VERYLOW, G_MID, G_HIGH, G_JUMPDEST = 3, 8, 10, 1
G_COLDSLOAD, G_WARMACCESS, G_SSET, G_SRESET = 2100, 100, 20000, 2900
MOD = 2**256

OPS = {0x00: "STOP", 0x01: "ADD", 0x02: "MUL", 0x03: "SUB", 0x54: "SLOAD", 0x55: "SSTORE",
       0x56: "JUMP", 0x57: "JUMPI", 0x5b: "JUMPDEST", 0x60: "PUSH1", 0x80: "DUP1"}

class OutOfGas(Exception):
    def __init__(self, msg, steps=0):
        super().__init__(msg); self.steps = steps      # also carries the number of instructions run before stopping

def run(code: bytes, storage: dict, gas: int, trace: bool = False):
    """Runs code from address 0. On success: (remaining gas, new storage, number of instructions run). Out of gas: OutOfGas.
    Storage is changed in a copy, so if an exception is raised, the caller "reverts" simply by keeping the original storage."""
    st = dict(storage); stack = []; pc = 0; touched = set(); steps = 0
    def charge(c):
        nonlocal gas
        if gas < c:
            raise OutOfGas(f"pc={pc}: needs {c} gas, {gas} gas left", steps)
        gas -= c
    while pc < len(code):
        name = OPS.get(code[pc])
        if name is None:
            raise OutOfGas(f"pc={pc}: unknown instruction 0x{code[pc]:02x}")   # the real EVM has a separate exception; simplified here
        steps += 1
        if trace:
            print(f"    pc={pc:>2} {name:<8} stack={stack} gas={gas}")
        if name == "STOP":
            break
        elif name == "PUSH1":
            charge(G_VERYLOW); stack.append(code[pc + 1]); pc += 2; continue
        elif name == "DUP1":
            charge(G_VERYLOW); stack.append(stack[-1])
        elif name in ("ADD", "MUL", "SUB"):
            charge(G_VERYLOW); b, a = stack.pop(), stack.pop()
            stack.append({"ADD": (a + b) % MOD, "MUL": (a * b) % MOD, "SUB": (a - b) % MOD}[name])
        elif name == "SLOAD":
            key = stack.pop(); charge(G_WARMACCESS if key in touched else G_COLDSLOAD); touched.add(key)
            stack.append(st.get(key, 0))
        elif name == "SSTORE":
            key, val = stack.pop(), stack.pop()
            cost = 0 if key in touched else G_COLDSLOAD
            touched.add(key)
            cost += G_SSET if (st.get(key, 0) == 0 and val != 0) else G_SRESET
            charge(cost)
            st[key] = val
        elif name == "JUMPDEST":
            charge(G_JUMPDEST)
        elif name == "JUMP":
            charge(G_MID); pc = stack.pop(); continue
        elif name == "JUMPI":
            charge(G_HIGH); dest, cond = stack.pop(), stack.pop()
            if cond != 0:
                pc = dest; continue
        pc += 1
    return gas, st, steps

# Bytecode imitating the core storage operation of OneNumber.set(42): PUSH1 42, PUSH1 0, SSTORE, STOP  (writes 42 to slot 0; ABI decoding and RETURN are left out)
set42 = bytes([0x60, 42, 0x60, 0, 0x55, 0x00])
gas_left, st, n = run(set42, {}, gas=30_000, trace=True)
print(f"write 42 to empty storage: {n} instructions, gas used {30_000 - gas_left} (= 3 + 3 + 2100 + 20000), storage {st}")

set7 = bytes([0x60, 7, 0x60, 0, 0x55, 0x00])
gas_left2, st2, n2 = run(set7, st, gas=30_000)
print(f"overwrite 42 with 7:       {n2} instructions, gas used {30_000 - gas_left2} (= 3 + 3 + 2100 + 2900), storage {st2}")

# Imitating the core operation of OneNumber.get(): PUSH1 0, SLOAD, STOP  — reading costs far less than writing
get = bytes([0x60, 0, 0x54, 0x00])
gas_left3, _, _ = run(get, st2, gas=30_000)
print(f"read slot 0:               gas used {30_000 - gas_left3} (= 3 + 2100)")

    pc= 0 PUSH1    stack=[] gas=30000
    pc= 2 PUSH1    stack=[42] gas=29997
    pc= 4 SSTORE   stack=[42, 0] gas=29994
    pc= 5 STOP     stack=[] gas=7894
write 42 to empty storage: 4 instructions, gas used 22106 (= 3 + 3 + 2100 + 20000), storage {0: 42}
overwrite 42 with 7:       4 instructions, gas used 5006 (= 3 + 3 + 2100 + 2900), storage {0: 7}
read slot 0:               gas used 2103 (= 3 + 2100)


In [ ]:
# NB13-2: applying a transaction — validity checks, up-front payment, execution, revert, refund, burning the base fee and paying the priority fee (for study)
G_TRANSACTION, G_TXDATAZERO, G_TXDATANONZERO = 21000, 4, 16

def intrinsic_gas(data: bytes) -> int:                       # the part of Yellow Paper equation (64) for an ordinary transaction (no contract creation, no access list)
    return G_TRANSACTION + sum(G_TXDATAZERO if b == 0 else G_TXDATANONZERO for b in data)

def apply_tx(state: dict, tx: dict, base_fee: int, beneficiary: str):
    """state: {addr: {"nonce","balance","code"(bytes or None),"storage"(dict)}}.
    tx: {"from","to","value","data","gas_limit","max_fee","max_priority","nonce"}.  All units are toy values."""
    S = {a: {**acc, "storage": dict(acc.get("storage", {}))} for a, acc in state.items()}
    sender = S[tx["from"]]
    g0 = intrinsic_gas(tx["data"])
    # --- intrinsic validity ((3)(5)(6)(7)(8) of Yellow Paper §6) ---
    if tx["nonce"] != sender["nonce"]:                  return S, f"invalid: nonce {tx['nonce']} ≠ account nonce {sender['nonce']}"
    if tx["gas_limit"] < g0:                            return S, f"invalid: gas limit {tx['gas_limit']} < intrinsic gas {g0}"
    if tx["max_fee"] < base_fee:                        return S, f"invalid: max fee {tx['max_fee']} < base fee {base_fee}"
    if tx["max_priority"] > tx["max_fee"]:              return S, "invalid: priority fee > max fee"
    v0 = tx["gas_limit"] * tx["max_fee"] + tx["value"]
    if sender["balance"] < v0:                          return S, f"invalid: balance {sender['balance']} < up-front cost {v0}"
    # --- effective price (equations (66)(67)) and irreversible up-front payment (equations (73)(74)) ---
    priority = min(tx["max_priority"], tx["max_fee"] - base_fee)
    price = base_fee + priority
    sender["nonce"] += 1
    sender["balance"] -= tx["gas_limit"] * price
    gas = tx["gas_limit"] - g0
    # --- execution ---
    to = S.setdefault(tx["to"], {"nonce": 0, "balance": 0, "code": None, "storage": {}})
    status = "success"
    try:
        if to["code"]:
            gas, new_storage, _ = run(to["code"], to["storage"], gas)
            to["storage"] = new_storage
        sender["balance"] -= tx["value"]; to["balance"] += tx["value"]
    except OutOfGas as e:
        status = f"out of gas ({e}) → execution reverted"
        gas = 0                                           # discard the storage and transfer changes (original state kept). The up-front payment and the nonce increment remain
    # --- refund and distribution (the refund counter Ar of equation (82) is 0 in this toy) ---
    gas_used = tx["gas_limit"] - gas
    sender["balance"] += gas * price                       # unused gas is returned at the price it was bought for
    S.setdefault(beneficiary, {"nonce": 0, "balance": 0, "code": None, "storage": {}})["balance"] += gas_used * priority
    burned = gas_used * base_fee
    return S, f"{status} | gas used {gas_used} | paid by sender {gas_used * price} = burned {burned} + validator {gas_used * priority}"

state = {"Alice": {"nonce": 0, "balance": 10_000_000, "code": None, "storage": {}},
         "Store": {"nonce": 1, "balance": 0, "code": set42, "storage": {}}}
BASE = 10                                                # toy base fee (per gas)
tx1 = {"from": "Alice", "to": "Store", "value": 0, "data": b"", "gas_limit": 50_000, "max_fee": 15, "max_priority": 2, "nonce": 0}
state, msg = apply_tx(state, tx1, BASE, "Val")
print("tx1 set(42):", msg)
print("   Store.storage =", state["Store"]["storage"], "| Alice balance", state["Alice"]["balance"], "| Val balance", state["Val"]["balance"])

tx2 = {**tx1, "nonce": 1, "gas_limit": 21_500}            # above the intrinsic gas of 21000, but not enough to reach SSTORE
state, msg = apply_tx(state, tx2, BASE, "Val")
print("tx2 gas limit 21,500:", msg)
print("   Store.storage =", state["Store"]["storage"], "(unchanged) | Alice balance", state["Alice"]["balance"], "(the fee was paid) | nonce", state["Alice"]["nonce"])

tx3 = {**tx1, "nonce": 2, "max_fee": 9}                   # a transaction that writes a max fee below the base fee of 10
_, msg = apply_tx(state, tx3, BASE, "Val")
print("tx3 max fee 9:", msg)

tx1 set(42): success | gas used 43106 | paid by sender 517272 = burned 431060 + validator 86212
   Store.storage = {0: 42} | Alice balance 9482728 | Val balance 86212
tx2 gas limit 21,500: out of gas (pc=4: needs 5000 gas, 494 gas left) → execution reverted | gas used 21500 | paid by sender 258000 = burned 215000 + validator 43000
   Store.storage = {0: 42} (unchanged) | Alice balance 9224728 (the fee was paid) | nonce 2
tx3 max fee 9: invalid: max fee 9 < base fee 10


In [ ]:
# NB13-3: infinite loop — gas stops code that does not stop (for study)
# JUMPDEST(0) ; PUSH1 0 ; JUMP  → jumps back to address 0 forever. 1 + 3 + 8 = 12 gas per loop.
forever = bytes([0x5b, 0x60, 0x00, 0x56])
for budget in [100, 1_000, 100_000]:
    try:
        run(forever, {}, gas=budget)
        print("finished?!")                                # this line should never run
    except OutOfGas as e:
        print(f"gas {budget:>7}: stopped after {e.steps:>6} instructions (about {e.steps // 3:>5} loops) — {e}")

state2 = {"Alice": {"nonce": 0, "balance": 10_000_000, "code": None, "storage": {}},
          "Loop": {"nonce": 1, "balance": 0, "code": forever, "storage": {}}}
tx = {"from": "Alice", "to": "Loop", "value": 0, "data": b"", "gas_limit": 100_000, "max_fee": 15, "max_priority": 2, "nonce": 0}
state2, msg = apply_tx(state2, tx, BASE, "Val")
print("transaction to the infinite-loop contract:", msg)
print("   Alice balance", state2["Alice"]["balance"], "— the full price of the gas limit (100,000 × 12 = 1,200,000) was paid. The nodes did only 100,000 gas worth of work")

gas     100: stopped after     27 instructions (about     9 loops) — pc=3: needs 8 gas, 0 gas left
gas    1000: stopped after    252 instructions (about    84 loops) — pc=3: needs 8 gas, 0 gas left
gas  100000: stopped after  25002 instructions (about  8334 loops) — pc=3: needs 8 gas, 0 gas left
transaction to the infinite-loop contract: out of gas (pc=3: needs 8 gas, 0 gas left) → execution reverted | gas used 100000 | paid by sender 1200000 = burned 1000000 + validator 200000
   Alice balance 8800000 — the full price of the gas limit (100,000 × 12 = 1,200,000) was paid. The nodes did only 100,000 gas worth of work
